# Lab 11 · Kỷ luật đo lường cho LLM: schema, nhãn tay & hậu kiểm

**Lập trình xử lý dữ liệu (LTXLDL) · 2627-1 · Giờ thực hành · Bài 11**

> 💡 File → **Save a copy in Drive** trước khi sửa.

Notebook bài giảng bài 11 chạy toàn bộ pipeline Gemini trên 24 đánh giá. Lab này tập trung vào
ba kỹ năng cốt lõi của hợp phần LLM và không yêu cầu khóa API: bạn sẽ chọn mẫu phù hợp,
viết schema, rồi **đánh giá một bộ đầu ra LLM mô phỏng có lỗi cài sẵn** bằng nhãn tay.
Đây cũng là quy trình mỗi nhóm sẽ thực hiện với ít nhất 100 nhãn trong bài tập lớn.

*Lab khoảng 50 phút; 30 phút cuối dành cho phần hỗ trợ bài tập lớn ở cuối notebook.*

**Mục tiêu bài lab**

1. Chọn mẫu đánh giá phù hợp để gửi LLM và ước lượng chi phí token trước khi gọi.
2. Viết schema Pydantic có enum và kiểm tra cách schema chặn nhãn bịa **trước khi** vào bảng.
3. Đo độ chính xác của đầu ra LLM trên nhãn tay và phân loại lỗi ngữ nghĩa.
4. Viết một quy tắc hậu kiểm tự động.

Phiên bản đề: `llm-eval-v1` (25/09/2026). Lưu đúng tên **`lab-11.ipynb`**.

## Cách làm và nộp bài

- Với phần khởi động và bài có hướng dẫn, bạn nên **tự gõ, không dùng AI**. Các bài kiểm tra định kỳ 🚫 đóng
  ở giờ lý thuyết sẽ kiểm tra những kỹ năng này. Bài tự làm cuối lab được gắn nhãn ✅ mở: được dùng AI, kèm khai báo.
- Điền thân hàm/lớp TODO và phần trả lời Markdown. Giữ tên hàm, tham số, cell ID và metadata;
  viết helper/import thêm trong chính cell bài làm.
- Grader chỉ chạy cell câu đó với input riêng (mẫu khác, bộ đầu ra LLM khác, id dạng chuỗi…), không phụ thuộc
  biến `rv`, `REVIEWS`, `GOLD`, `LLM_OUT`, `hop_le` ở cell khác. `pd` và `np` được cung cấp sẵn.
  Riêng **Bài 4**, grader cung cấp sẵn `ReviewInfo` **chuẩn** (đúng hợp đồng Bài 3) và `ValidationError`,
  nên Bài 4 được chấm độc lập với Bài 3 — không định nghĩa lại `ReviewInfo` trong cell Bài 4.
- Public checks chỉ là ví dụ tự kiểm tra. Một check lỗi không dừng các check sau.
  TODO báo **CHƯA LÀM**; lỗi cú pháp vẫn cần sửa. Không sửa input; không ghi cứng kết quả.
- Nếu chưa tiến triển sau 3 phút ở một bước, hãy trao đổi với giảng viên thực hành.
- **Restart & Run all**, lưu notebook, commit/push vào repo của bạn trong org và tạo tag theo cuối bài.
  **Không bao giờ** dán khóa API vào notebook.

## Rubric dự kiến — 100 điểm

| Mục | Điểm | Cách đánh giá |
|---|---:|---|
| Bài 1 (chọn mẫu), Bài 2 (ước chi phí) — mỗi bài 10 điểm | 20 | Tự động |
| Bài 3 (schema), 4 (kiểm tra hàng loạt), 5 (độ chính xác), 6 (hậu kiểm) — mỗi bài 15 điểm | 60 | Tự động |
| Diễn giải: lỗi schema, phân loại lỗi ngữ nghĩa, vai trò nhãn tay và hậu kiểm | 15 | Giảng viên |
| Khai báo AI và bằng chứng kiểm chứng | 5 | Giảng viên |

Phần 🧭 hỗ trợ bài tập lớn ở cuối lab làm theo nhóm, không tính vào điểm lab.

In [ ]:
# Public checks giúp tự kiểm tra; đây không phải điểm chính thức.
# Không sửa cell này trong bài nộp.
PUBLIC_RESULTS = {}

def public_check(case_id, check):
    try:
        check()
    except NotImplementedError:
        status, detail = "CHƯA LÀM", "Điền phần TODO rồi chạy lại."
    except AssertionError as exc:
        status, detail = "CHƯA ĐẠT", str(exc) or "Kết quả chưa khớp ví dụ công khai."
    except Exception as exc:
        status, detail = "LỖI", f"{type(exc).__name__}: {exc}"
    else:
        status, detail = "ĐẠT", "Ví dụ công khai đã qua; chưa đại diện toàn bộ rubric."
    PUBLIC_RESULTS[case_id] = status
    print(f"[{status}] {case_id}: {detail}")

def require_answer(value):
    if value is None or value is Ellipsis:
        raise NotImplementedError

def preview(label, action):
    try:
        value = action()
    except NotImplementedError:
        print(f"{label}: chưa chạy được vì còn TODO.")
    except Exception as exc:
        print(f"{label}: {type(exc).__name__}: {exc}")
    else:
        print(label)
        print(value)

def show_public_summary():
    print("PUBLIC CHECKS — không phải điểm chính thức")
    for case_id, status in PUBLIC_RESULTS.items():
        print(f"{case_id}: {status}")
    print("Sau khi sửa, Restart & Run all để làm mới toàn bộ kết quả.")

## Dữ liệu và thư viện

Lab cần `pydantic` (bản 2, Colab có sẵn). Nếu cell dưới báo thiếu, chạy `%pip install pydantic` trong một cell riêng rồi Restart.

Mặc định `USE_SNAPSHOT = False`: bước chọn mẫu dùng **240 bình luận giả lập** (`DEMO_RV`). Đặt `USE_SNAPSHOT = True`
để đọc cột `comments` của `data/reviews.csv.gz` (bảng đầy đủ) của snapshot Santiago **2026-06-29** — ô đó tải ~59MB
rồi mới lấy mẫu, chờ 10–30 giây, **không** phải treo. Phần 2 luôn dùng 10 đánh giá thật, nhãn tay `GOLD`
và 10 đầu ra LLM mô phỏng có sẵn trong notebook.

In [ ]:
import json

import numpy as np
import pandas as pd

try:
    import pydantic
    print("pydantic", pydantic.VERSION)
except ImportError:
    print("Thiếu pydantic: chạy %pip install pydantic trong một cell riêng rồi Restart.")

URL = ("https://data.insideairbnb.com/chile/rm/santiago/"
       "2026-06-29/data/reviews.csv.gz")

# 240 bình luận GIẢ LẬP (cột id, comments) để thử bước chọn mẫu khi chưa tải snapshot.
_MAU_CAU = [
    "Great location, clean apartment and a very responsive host.",
    "Buena ubicación y departamento limpio, volvería sin duda.",
    "Ok",
    ".",
    "Nice",
    "Todo bien, gracias.",
    "The apartment was noisy at night but the view was amazing.",
    "Excelente anfitrión, muy atento a todo lo que necesitamos.",
    "Muy bueno",
    "Good value for money, close to the metro and supermarkets.",
]
DEMO_RV = pd.DataFrame({
    "id": range(5000, 5240),
    "comments": [f"{_MAU_CAU[k % 10]} (#{k})" for k in range(240)],
})

# Đặt USE_SNAPSHOT = True để chọn mẫu từ 690 nghìn đánh giá Santiago.
USE_SNAPSHOT = False

if USE_SNAPSHOT:
    rv = pd.read_csv(URL, usecols=["id", "comments"]).dropna(subset=["comments"])
    DATA_LABEL = "Snapshot Santiago 2026-06-29 — cột comments"
else:
    rv = DEMO_RV.copy()
    DATA_LABEL = "GIẢ LẬP — 240 bình luận"
rv["comments"] = rv["comments"].str.replace("<br/>", " ", regex=False)
print(DATA_LABEL, "·", len(rv), "bình luận")

## Phần 0 · Chọn mẫu & ước chi phí

Gói miễn phí có giới hạn, vì vậy không nên gửi toàn bộ 690 nghìn đánh giá cho mô hình. Chọn mẫu là một bước cần thiết
của pipeline; lab 7 cho thấy khoảng 59 nghìn bình luận quá ngắn để phân tích.

## Bài 1 · Chọn mẫu tái lập được

### Hợp đồng hàm · 10 điểm tự động

`choose_sample(rv: pd.DataFrame, do_dai_min: int = 50, n: int = 100, seed: int = 42) -> pd.DataFrame`

Nhận bảng có cột `comments` (chuỗi, không thiếu) cùng các cột khác. Lọc các dòng có độ dài bình luận **>= `do_dai_min`**,
rồi lấy mẫu **`n` dòng** bằng `.sample(n, random_state=seed)` trên bảng đã lọc. Trả DataFrame mẫu (đủ mọi cột, giữ index gốc,
thứ tự do `sample` trả về). Có ít nhất `n` dòng đủ dài. Không sửa input.

In [ ]:
def choose_sample(rv: pd.DataFrame, do_dai_min: int = 50, n: int = 100, seed: int = 42) -> pd.DataFrame:
    # TODO: triển khai đúng hợp đồng ở trên.
    raise NotImplementedError

In [ ]:
def check_q1():
    mau = choose_sample(DEMO_RV)
    assert isinstance(mau, pd.DataFrame) and len(mau) == 100
    assert list(mau.columns) == ["id", "comments"]
    assert mau["comments"].str.len().min() >= 50
    assert mau["id"].is_unique
    assert mau["id"].head(3).tolist() == [5089, 5096, 5009], "Lọc trước, sample(n, random_state=seed) sau"
    pd.testing.assert_frame_equal(mau, choose_sample(DEMO_RV), obj="Cùng seed phải cho cùng mẫu")
    assert len(choose_sample(DEMO_RV, do_dai_min=60, n=5, seed=1)) == 5

public_check("Q1 · ví dụ công khai", check_q1)

## Bài 2 · Ước lượng token và chi phí trước khi gọi

Quy tắc ngón tay cái: **~4 ký tự/token** — đủ cho ước lượng chi phí. Giá flash-lite: **$0,25 / 1 triệu token vào**.

### Hợp đồng hàm · 10 điểm tự động

`estimate_cost(comments: pd.Series, gia_moi_trieu: float = 0.25) -> dict`

Nhận Series bình luận (chuỗi) và giá USD cho 1 triệu token vào. Trả dict có đúng các khóa:
`tong_token` — int, **tổng của `len(bình luận) // 4`** trên từng bình luận (chia nguyên từng dòng rồi mới cộng);
`chi_phi` — float, `tong_token × gia_moi_trieu / 1_000_000` (USD, không làm tròn). Chưa tính token của prompt.

In [ ]:
def estimate_cost(comments: pd.Series, gia_moi_trieu: float = 0.25) -> dict:
    # TODO: triển khai đúng hợp đồng ở trên.
    raise NotImplementedError

In [ ]:
def check_q2():
    result = estimate_cost(pd.Series(["abcdefg", "abcd", "abc"]))
    assert set(result) == {"tong_token", "chi_phi"}
    assert result["tong_token"] == 2, "7//4 + 4//4 + 3//4 = 1 + 1 + 0 (không phải 14//4)"
    assert result["chi_phi"] == 2 * 0.25 / 1_000_000
    mau = DEMO_RV[DEMO_RV["comments"].str.len() >= 50].sample(100, random_state=42)
    assert estimate_cost(mau["comments"])["tong_token"] == 1589
    assert estimate_cost(mau["comments"], 1.0)["chi_phi"] < 0.01

public_check("Q2 · ví dụ công khai", check_q2)

In [ ]:
def thuc_hanh_q1_q2():
    global mau
    mau = choose_sample(rv)
    kq = estimate_cost(mau["comments"])
    so_du_dai = int((rv["comments"].str.len() >= 50).sum())
    return (f"Chọn 100/{so_du_dai:,} đánh giá đủ dài — mẫu tái lập được nhờ random_state.\n"
            f"~{kq['tong_token']:,} token dữ liệu ≈ ${kq['chi_phi']:.5f} — chưa tính prompt.")

preview("Bài 1–2 trên dữ liệu đang dùng", thuc_hanh_q1_q2)

## Phần 1 · Schema là tầng kiểm tra đầu tiên

## Bài 3 · Viết schema Pydantic có enum

### Hợp đồng lớp · 15 điểm tự động

`class ReviewInfo(BaseModel)` với **đúng 4 trường, không có giá trị mặc định**:

- `sentiment`: `Literal["positive", "mixed", "negative"]`;
- `aspects_positive`, `aspects_negative`: `list[Aspect]` với `Aspect` là 6 khía cạnh đã cho sẵn trong cell;
- `language`: `str`.

Grader tạo `ReviewInfo(**du_lieu)` với các bộ dữ liệu khác nhau: dữ liệu hợp lệ phải cho đúng giá trị các trường
(`model_dump()`); nhãn ngoài enum, trường thiếu, `language` không phải chuỗi hay khía cạnh không phải list phải
**ném `ValidationError`**. Không thêm validator tự sửa dữ liệu (ví dụ tự đổi chữ hoa thành chữ thường).

In [ ]:
from typing import Literal

from pydantic import BaseModel, ValidationError

Aspect = Literal["location", "cleanliness", "host", "noise", "amenities", "value"]


class ReviewInfo(BaseModel):
    # TODO: khai báo 4 trường theo hợp đồng ở trên (thay dòng pass).
    pass

In [ ]:
def check_q3():
    if not ReviewInfo.model_fields:
        raise NotImplementedError
    ok = ReviewInfo.model_validate_json(
        '{"sentiment": "mixed", "aspects_positive": ["location"], '
        '"aspects_negative": ["noise"], "language": "en"}')
    assert ok.sentiment == "mixed" and ok.aspects_negative == ["noise"]
    assert set(ReviewInfo.model_fields) == {"sentiment", "aspects_positive", "aspects_negative", "language"}
    for sai in ['{"sentiment": "happy", "aspects_positive": [], "aspects_negative": [], "language": "en"}',
                '{"sentiment": "mixed", "aspects_positive": ["photos"], "aspects_negative": [], "language": "en"}',
                '{"sentiment": "mixed", "aspects_positive": [], "aspects_negative": []}']:
        try:
            ReviewInfo.model_validate_json(sai)
        except ValidationError:
            continue
        raise AssertionError(f"Lẽ ra phải bị chặn: {sai}")

public_check("Q3 · ví dụ công khai", check_q3)

## Phần 2 · Đánh giá đầu ra LLM bằng nhãn tay

Dưới đây là **10 đánh giá thật** tại Santiago, bộ nhãn tay `GOLD` và **10 đầu ra LLM mô phỏng**.
Một số lỗi đã được cài sẵn để mô phỏng dữ liệu thực tế. Nhiệm vụ của bạn là dùng schema
và nhãn tay để phát hiện các lỗi này.

In [ ]:
REVIEWS = {
 1: "Place is great for a family, all clean, good location, bit noisy as the main avenue is behind, but we had a great time.",
 2: "Apartment was irrelevant with pictures and not clean too so we cancelled our reservation. Alvaro helped us about cancellation process.",
 3: "Cristian fue muy amable en todo momento, el lugar como se describia. La zona con muy buena movilidad. Muy recomendable. Gracias Cristian",
 4: "Hermoso alojamiento! Lo pasamos re bien mi hija y yo. Es un poco ruidosa la zona si se abre la ventana. La vista es bella y esta muy bien ubicado. Sin duda volveriamos",
 5: "Great location but apartment needs some attention to detail. Cable TV and wifi was out of service. Communication with host was poor. Bedding was not optimal.",
 6: "Good WiFi, great location and centrally located. Shower was hot and had good pressure and there was enough space for two people for 6 days.",
 7: "Es tal cual las fotos, buena ubicacion, tranquilo y sin ruido. Es en un piso 15 por si le temen a las alturas.",
 8: ".",
 9: "Nice place in cool region. Very noisy environment and apartment is not very clean.",
 10: "location was great, wifi was spotty. But overall not a bad place.",
}

# Nhãn tay của nhóm (GOLD) — quy ước: đánh giá rỗng/vô nghĩa -> mixed, không khía cạnh, "und"
GOLD = {
 1: {"sentiment": "mixed",    "language": "en"},
 2: {"sentiment": "negative", "language": "en"},
 3: {"sentiment": "positive", "language": "es"},
 4: {"sentiment": "mixed",    "language": "es"},
 5: {"sentiment": "negative", "language": "en"},
 6: {"sentiment": "positive", "language": "en"},
 7: {"sentiment": "positive", "language": "es"},
 8: {"sentiment": "mixed",    "language": "und"},
 9: {"sentiment": "mixed",    "language": "en"},
 10: {"sentiment": "mixed",   "language": "en"},
}

# Đầu ra LLM mô phỏng (JSON thô) — CÓ LỖI CÀI SẴN, không sửa
LLM_OUT = {
 1: '{"sentiment": "mixed", "aspects_positive": ["cleanliness", "location"], "aspects_negative": ["noise"], "language": "en"}',
 2: '{"sentiment": "negative", "aspects_positive": ["host"], "aspects_negative": ["photos", "cleanliness"], "language": "en"}',
 3: '{"sentiment": "positive", "aspects_positive": ["host", "location"], "aspects_negative": [], "language": "es"}',
 4: '{"sentiment": "positive", "aspects_positive": ["location"], "aspects_negative": [], "language": "es"}',
 5: '{"sentiment": "negative", "aspects_positive": ["location"], "aspects_negative": ["amenities", "host"]}',
 6: '{"sentiment": "positive", "aspects_positive": ["amenities", "location"], "aspects_negative": [], "language": "en"}',
 7: '{"sentiment": "positive", "aspects_positive": ["location"], "aspects_negative": [], "language": "es"}',
 8: '{"sentiment": "mixed", "aspects_positive": [], "aspects_negative": [], "language": "und"}',
 9: '{"sentiment": "negative", "aspects_positive": ["location"], "aspects_negative": ["noise", "cleanliness"], "language": "en"}',
 10: '{"sentiment": "negative", "aspects_positive": ["location"], "aspects_negative": ["amenities"], "language": "en"}',
}
print(len(REVIEWS), "đánh giá · 10 đầu ra mô phỏng · 10 nhãn tay")

## Bài 4 · Kiểm tra hàng loạt — schema phát hiện được gì?

### Hợp đồng hàm · 15 điểm tự động

`validate_outputs(llm_out: dict) -> dict`

Nhận dict `{id: chuỗi JSON thô}` (id có thể là số hoặc chuỗi — **giữ nguyên**). Với từng phần tử **theo thứ tự của dict**,
gọi `ReviewInfo.model_validate_json(...)`: hợp lệ → đưa đối tượng vào dict `hop_le[id]`; ném `ValidationError`
(nhãn ngoài enum, thiếu trường, JSON hỏng…) → đưa id vào list `loi_schema`. Chỉ bắt `ValidationError`.
Trả dict có đúng các khóa `hop_le`, `loi_schema`. Không sửa input.
Khi chấm, `ReviewInfo` chuẩn và `ValidationError` có sẵn; trong notebook, cell dùng lớp bạn viết ở Bài 3.

In [ ]:
def validate_outputs(llm_out: dict) -> dict:
    # TODO: triển khai đúng hợp đồng ở trên.
    raise NotImplementedError

In [ ]:
def check_q4():
    result = validate_outputs(LLM_OUT)
    assert set(result) == {"hop_le", "loi_schema"}
    hop_le, loi_schema = result["hop_le"], result["loi_schema"]
    assert len(hop_le) == 8 and loi_schema == [2, 5]
    assert all(isinstance(v, ReviewInfo) for v in hop_le.values())
    assert list(hop_le) == [1, 3, 4, 6, 7, 8, 9, 10] and hop_le[9].sentiment == "negative"
    assert validate_outputs({"x": "{khong phai json"})["loi_schema"] == ["x"], "JSON hỏng cũng là lỗi schema"

public_check("Q4 · ví dụ công khai", check_q4)

In [ ]:
def ly_do_bi_chan():
    global hop_le, loi_schema
    kq = validate_outputs(LLM_OUT)
    hop_le, loi_schema = kq["hop_le"], kq["loi_schema"]
    dong = [f"Schema chặn {len(loi_schema)} đầu ra: ID {loi_schema}"]
    for rid in loi_schema:
        try:
            ReviewInfo.model_validate_json(LLM_OUT[rid])
        except ValidationError as e:
            dong.append(f"--- id {rid}: {e.errors()[0]['type']} tại {e.errors()[0]['loc']}")
    return "\n".join(dong)

preview("Bài 4 — lý do từng đầu ra bị chặn", ly_do_bi_chan)

**Schema chặn gì?** mỗi đầu ra bị chặn sai ở đâu? Trong pipeline thật, nên xử lý các đầu ra này thế nào? …

## Bài 5 · Độ chính xác của nhãn cảm xúc

### Hợp đồng hàm · 15 điểm tự động

`sentiment_accuracy(du_doan: dict, gold: dict) -> dict`

Nhận dict `{id: nhãn cảm xúc do LLM dự đoán}` (chỉ các đầu ra hợp lệ, ít nhất một phần tử) và dict nhãn tay
`{id: {"sentiment": ..., ...}}` chứa mọi id của `du_doan`. Trả dict có đúng các khóa:
`so_dung` (int, số id dự đoán trùng `gold[id]["sentiment"]`), `acc` (float, `so_dung / len(du_doan)`) và
`sai` (list id dự đoán sai, **theo thứ tự của `du_doan`**). Mẫu số chỉ gồm các đầu ra hợp lệ. Không sửa input.

In [ ]:
def sentiment_accuracy(du_doan: dict, gold: dict) -> dict:
    # TODO: triển khai đúng hợp đồng ở trên.
    raise NotImplementedError

In [ ]:
def check_q5():
    du_doan = {rid: json.loads(raw)["sentiment"] for rid, raw in LLM_OUT.items() if rid not in (2, 5)}
    result = sentiment_accuracy(du_doan, GOLD)
    assert set(result) == {"so_dung", "acc", "sai"}
    assert result["so_dung"] == 5 and result["acc"] == 0.625
    assert result["sai"] == [4, 9, 10]
    assert sentiment_accuracy({"a": "mixed"}, {"a": {"sentiment": "mixed"}, "b": {"sentiment": "negative"}})["acc"] == 1.0

public_check("Q5 · ví dụ công khai", check_q5)

In [ ]:
def thuc_hanh_q5():
    kq = sentiment_accuracy({rid: m.sentiment for rid, m in hop_le.items()}, GOLD)
    dong = [f"Accuracy sentiment: {kq['acc']:.0%} — sai ở id {kq['sai']}."]
    for rid in kq["sai"]:
        dong.append(f"[{rid}] GOLD={GOLD[rid]['sentiment']:8s} LLM={hop_le[rid].sentiment:8s} | {REVIEWS[rid][:90]}")
    return "\n".join(dong)

preview("Bài 5 trên 8 đầu ra hợp lệ", thuc_hanh_q5)

### Nhìn vào chỗ sai — phân loại lỗi

Đọc lại ba đánh giá bị gán sai ở output trên, rồi đối chiếu bảng phân loại:

| ID | GOLD | LLM | Kiểu lỗi |
|---|---|---|---|
| 4 | mixed | positive | **bỏ sót ý chê** — lời khen dài che khuất nhận xét về tiếng ồn |
| 9 | mixed | negative | **trường hợp khó phân định** — nhóm gán mixed ("nice place" + 2 ý chê) |
| 10 | mixed | negative | **hiểu sai phủ định** — "not a *bad* place" là khen nhẹ, mô hình lại đọc thành chê |

Trường hợp 9 cho thấy người gán nhãn cũng có thể không thống nhất với nhau. Vì vậy, hướng dẫn
gán nhãn đã được nhóm thống nhất trước là **mốc tham chiếu** và phải được áp dụng nhất quán.

**Độ chính xác và mẫu số:** 62,5% được tính trên bao nhiêu đầu ra? Nếu tính trên cả 10 (coi 2 đầu ra bị chặn là sai) thì sao — nên báo cáo thế nào? …

## Bài 6 · Hậu kiểm tự động

Quy tắc hậu kiểm: đầu ra nào **tự mâu thuẫn** — `sentiment` là `"positive"` mà `aspects_negative` không rỗng,
hoặc `"negative"` mà `aspects_positive` không rỗng?

### Hợp đồng hàm · 15 điểm tự động

`contradictions(outputs: dict) -> list`

Nhận dict `{id: dict có các khóa sentiment, aspects_positive, aspects_negative}` (ví dụ `m.model_dump()` của các đầu ra hợp lệ).
Trả list id vi phạm quy tắc trên, **theo thứ tự của dict**. `"mixed"` không bao giờ vi phạm. Không cần nhãn tay, không sửa input.

In [ ]:
def contradictions(outputs: dict) -> list:
    # TODO: triển khai đúng hợp đồng ở trên.
    raise NotImplementedError

In [ ]:
def check_q6():
    outputs = {rid: json.loads(raw) for rid, raw in LLM_OUT.items() if rid not in (2, 5)}
    result = contradictions(outputs)
    assert isinstance(result, list) and result == [9, 10]
    assert contradictions({"a": {"sentiment": "positive", "aspects_positive": [], "aspects_negative": ["noise"]},
                           "b": {"sentiment": "mixed", "aspects_positive": ["host"], "aspects_negative": ["noise"]}}) == ["a"]

public_check("Q6 · ví dụ công khai", check_q6)

In [ ]:
def thuc_hanh_q6():
    mau_thuan = contradictions({rid: m.model_dump() for rid, m in hop_le.items()})
    return f"Hậu kiểm bắt id {mau_thuan} — so với các ca gán sai ở Bài 5."

preview("Bài 6 trên 8 đầu ra hợp lệ", thuc_hanh_q6)

**Nhãn tay và hậu kiểm:** quy tắc bắt được những ca sai nào ở Bài 5, bỏ sót ca nào? Vì sao pipeline cần cả nhãn tay lẫn hậu kiểm? …

## Bài tự làm ✅ mở · Chạy thật với khóa API của bạn — không cộng vào 80 điểm tự động

Nếu đã có khóa API, hãy dùng mã nguồn mẫu ở mục 4 của notebook bài giảng bài 11: đầu ra có cấu trúc
với `ReviewInfo.model_json_schema()`. Gọi Gemini cho **10 đánh giá trong `REVIEWS`**, rồi:
(1) kiểm tra schema như Bài 4; (2) đo độ chính xác với `GOLD` như Bài 5; (3) so sánh kết quả
của mô hình thật với bộ mô phỏng trong lab. Dùng `time.sleep(4)` giữa các lời gọi và lưu cache JSON.
Ghi lại các ID được gán đúng hoặc sai; mô hình có hiểu sai câu phủ định ở ID 10 không?
Khóa API chỉ đọc từ **Colab Secrets / biến môi trường**, không bao giờ dán vào notebook.

In [ ]:
# Viết bài tự làm của bạn ở đây (cần GEMINI_API_KEY trong Colab Secrets; không được chấm tự động).

**Bài tự làm của tôi:** …

## Khai báo sử dụng AI

- **Công cụ đã dùng:** … *(hoặc “không dùng”)*
- **Bài đã dùng AI và yêu cầu chính:** …
- **Một gợi ý đã kiểm chứng, cách kiểm chứng và kết quả:** …

Không chỉ ghi “code chạy được”; nêu id, nhãn hoặc phép đếm tay đã dùng để đối chiếu.

---

## 🧭 Hỗ trợ bài tập lớn (~30 phút — làm theo nhóm, không tính điểm lab)

Trọng tâm tuần này: **thiết lập hợp phần LLM có thể kiểm chứng và chạy lại được.**

1. ☐ Khóa API Gemini của người phụ trách nằm trong **Colab Secrets / biến môi trường** —
   tuyệt đối chưa từng xuất hiện trong code hay commit nào (soát bằng `git log -p | grep`).
2. ☐ **Mẫu đánh giá** để gán nhãn đã chốt: cách lọc theo độ dài và thời gian, cùng `random_state`,
   đủ ≥100, cân nhắc tỷ lệ ngôn ngữ.
3. ☐ **Schema + enum khía cạnh** bản nháp đã viết (theo lab này); danh mục khía cạnh
   hợp với câu hỏi phân tích của nhóm.
4. ☐ **Hướng dẫn gán nhãn** 5–10 dòng đã viết chung (quy ước trường hợp rỗng, mixed và đa
   ngôn ngữ); mỗi đánh giá do hai người gán độc lập, sau đó thảo luận các trường hợp bất đồng.
5. ☐ Phương pháp đối chứng không dùng LLM (từ khoá bài 7) chạy được trên đúng mẫu để so sánh.

> Nhóm hoàn thành sớm: chạy thử 5 lời gọi Gemini với schema của nhóm và kiểm tra xem đầu ra thực tế
> có đúng cấu trúc hay không.

## Tóm tắt bài lab

| Bạn đã làm | Dùng cho |
|---|---|
| Chọn mẫu + ước token trước khi gọi | kế hoạch chi phí cho hợp phần LLM |
| Schema/enum chặn nhãn bịa và trường thiếu | pipeline LLM của nhóm |
| Độ chính xác trên nhãn tay + phân loại lỗi | mục "đo chất lượng trên ≥100 nhãn" của đề |
| Hậu kiểm tự mâu thuẫn | vận hành pipeline sau khi nộp |

**Trước khi nộp:** hoàn thành TODO và câu trả lời, chạy Restart & Run all, lưu notebook.

Bài giảng tiếp theo sẽ dùng **trực quan hoá cơ bản** để biểu diễn các con số này.

## Thông tin và nộp bài

**Họ tên / MSSV / lớp:** [Điền]

1. Giữ tên file **`lab-11.ipynb`** ở gốc repo (dấu gạch ngang, không phải `lab_11.ipynb`).
2. Hoàn thành hàm/lớp TODO, các diễn giải và khai báo AI. Kiểm tra notebook **không chứa khóa API** trước khi commit.
3. Restart & Run all; đọc bảng public checks, lưu file rồi commit/push lên `main`.
4. Tạo tag `submit/lab-11/v1`; lần sửa tiếp dùng v2 hoặc v3, không di chuyển tag đã nộp.
5. Repo cần được giảng viên đăng ký chấm. Xem status `private-grader/lab-11` trên commit nộp;
   điểm tự động tối đa 80, còn 20 điểm chờ giảng viên. Xem lịch quét trong README.

Tag v1/v2/v3 là **lần nộp**, không phải phiên bản đề. Không đổi metadata để chọn bộ test khác.

In [ ]:
show_public_summary()